# ISNAD 24/7 Quran Live — Colab test
Short **test** stream (Arabic + Tamil + recitation) to YouTube, no OBS. Colab stops after a few hours, so use it only to test; run the Docker setup on your VPS for 24/7.
Run the cells top to bottom. Never share a notebook that contains your stream key.

In [ ]:
# 1) Settings
from getpass import getpass
YT_KEY = getpass("YouTube stream key: ")
REPO_URL = "https://github.com/YOUR_USER/YOUR_REPO.git"   # public repo, or leave and upload the zip in step 3
DURATION_MIN = 20                                          # test length in minutes
LIVE_QUERY = "surah=1&audio=1&en=1"                        # try surah=18, or surah=0 for all
W, H, FPS, VBR = 854, 480, 15, "1200k"

In [ ]:
# 2) Install tools (about 2-4 minutes)
import subprocess
def sh(c, **k):
    r = subprocess.run(c, shell=True, text=True, capture_output=True, **k)
    print((r.stdout + r.stderr)[-1500:]); return r
sh("apt-get update -qq && apt-get install -y -qq xvfb pulseaudio ffmpeg fonts-noto fonts-noto-core fonts-hosny-amiri nodejs")
sh("pip -q install playwright && PLAYWRIGHT_BROWSERS_PATH=/opt/pw playwright install --with-deps chromium")
sh("id live || useradd -m live; node -v")

In [ ]:
# 3) Get the code (clone, or upload isnad-api.zip)
import os
if os.path.exists("/content/isnad-api"): sh("rm -rf /content/isnad-api")
r = sh(f"git clone --depth 1 {REPO_URL} /content/isnad-api")
if r.returncode != 0:
    from google.colab import files
    up = files.upload()
    sh(f"cd /content && unzip -qo {list(up)[0]}")
sh("ls /content/isnad-api | head")

In [ ]:
# 4) Start the stream; it stops by itself after DURATION_MIN
import subprocess, time, os, textwrap
sh("pkill -f 'node server.js'; pkill Xvfb; pkill pulseaudio; pkill ffmpeg; pkill chrom; true")
env = dict(os.environ, PORT="3000")
srv = subprocess.Popen(["node","server.js"], cwd="/content/isnad-api", env=env)
time.sleep(2)
open("/tmp/chrome_open.py","w").write(textwrap.dedent(f'''
    import time
    from playwright.sync_api import sync_playwright
    with sync_playwright() as p:
        b = p.chromium.launch(headless=False, args=["--no-sandbox","--kiosk","--window-position=0,0","--window-size={W},{H}","--autoplay-policy=no-user-gesture-required"])
        pg = b.new_page(no_viewport=True)
        pg.goto("http://localhost:3000/live?{LIVE_QUERY}")
        time.sleep(10**6)
'''))
sh("chmod 644 /tmp/chrome_open.py")
U = "runuser -u live -- env XDG_RUNTIME_DIR=/tmp/xdg-live HOME=/home/live DISPLAY=:99 PLAYWRIGHT_BROWSERS_PATH=/opt/pw "
sh("mkdir -p /tmp/xdg-live && chown live /tmp/xdg-live && chmod 700 /tmp/xdg-live && chmod -R a+rX /opt/pw")
xvfb = subprocess.Popen(f"Xvfb :99 -screen 0 {W}x{H}x24", shell=True)
time.sleep(1)
sh(U + "pulseaudio --start --exit-idle-time=-1")
sh(U + "pactl load-module module-null-sink sink_name=live")
sh(U + "pactl set-default-sink live")
chrome = subprocess.Popen(U + "python3 /tmp/chrome_open.py", shell=True)
time.sleep(12)
cmd = (U + f"timeout {DURATION_MIN*60} ffmpeg -loglevel warning -f x11grab -framerate {FPS} -video_size {W}x{H} -i :99 "
       f"-f pulse -i live.monitor -c:v libx264 -preset veryfast -tune zerolatency -b:v {VBR} -maxrate {VBR} -bufsize 2400k -pix_fmt yuv420p -g {FPS*2} "
       f"-c:a aac -b:a 128k -ar 44100 -f flv rtmp://a.rtmp.youtube.com/live2/{YT_KEY}")
print("Streaming... check YouTube Studio -> Go live. Stop with the stop button on this cell.")
try:
    subprocess.run(cmd, shell=True)
finally:
    for pr in (chrome, xvfb, srv): pr.terminate()
    sh("pkill ffmpeg; pkill pulseaudio; true")